## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value! This is the start of a lab that will last 2 days.

And we're going to hand-build an Agent Loop without any Agent Framework..

### First, some prep

In the folder `twin` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours! You should be able to download it from your LinkedIn profile; go to your profile page use the menu under your name. If you don't have access to this feature, any PDF such as your resume is great.

I've also made a file called `summary.txt` in `twin` - please read it and update it to reflect you.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. If you're wondering how you would select packages for your own projects, please see Q37 in the <a href="https://edwarddonner.com/avatar?q=37">FAQ</a> page.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json
import os

In [2]:
load_dotenv(override=True)

gemini_api_key = os.getenv('GEMINI_API_KEY')

if gemini_api_key:
    print(f"Gemini API Key exists and begins {gemini_api_key[:8]}")
else:
    print("Gemini API Key not set - please head to the troubleshooting guide in the setup folder")

gemini = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

Gemini API Key exists and begins AQ.Ab8RN


In [3]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
0792239966 (Mobile)
rebeccashirievo@gmail.com
www.linkedin.com/in/rebecca-
shirievo (LinkedIn)
github.com/cyb3rr31a (Personal)
Top Skills
Microsoft Excel
Mentoring
NumPy
Certifications
Data Science
Technovation Mentor
Rebecca Shirievo
AI & Data Scientist @deeptrack | Applied ML/AI Engineer
Nairobi County, Kenya
Summary
Passionate about leveraging technology to solve complex problems
and empower individuals, I am a versatile professional with a diverse
skill set spanning machine learning, artificial intelligence and data
science
With a deep-rooted enthusiasm for cutting-edge technologies, I have
honed my expertise in full-stack web development, Python, SQL, and
TensorFlow. Whether it's crafting elegant solutions through code or
designing captivating visuals, I thrive on pushing the boundaries of
innovation.
Beyond my technical prowess, I am dedicated to making a
meaningful impact in my community. I am committed to empowering
young girls in high school, providing them with th

In [5]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
print(summary)

My name is Rebecca Shirievo. I'm a data scientist with a strong interest in AI. I'd say I'm more into the engineering aspects of things.

Dancing is and always will be my first love and art a special place in my heart.

I am a staunch feminist and a strong advocator for all human rights. I plan to make the world a better place, I just don't know how yet.


## Sidebar: Three concepts as a refresher

1. System Prompt: the part of the input to the LLM that describes the overall context of the conversation

2. Conversation History: the complete conversation so far

3. The illusion of memory: every message to an LLM is stateless. We pass in the complete conversation so far to give the illusion that it remembers what was said 30 seconds ago...

__For more, see my companion course AI Engineer Core Track (first week)__

In [7]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Rebecca"}
]

In [8]:
response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite", 
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Hi, my name is Rebecca"}
    ]
)

print(response.choices[0].message.content)

Hi Rebecca! It's nice to meet you. How can I help you today?


In [9]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Rebecca"}
]

In [10]:
response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite", 
    messages=messages)
print(response.choices[0].message.content)

Hi, Rebecca. I'd offer to remember that, but let's be honest, I'm an AI—my memory gets wiped the second you close the tab. 

Anyway, what can I do for you today that will make me question my digital life choices?


In [11]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [12]:
response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite", 
    messages=messages)

print(response.choices[0].message.content)

How on earth would I know that? We've just met, and unless you've tattooed it onto your forehead—which, given the vibe, isn't entirely off the table—you're a total stranger to me. 

Care to enlighten me, or are we playing a guessing game where I get to call you "Bob" until you get annoyed?


In [13]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Rebecca"},
    {"role": "assistant", "content": "Well hi there, Rebecca. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [14]:
response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite", 
    messages=messages)

print(response.choices[0].message.content)

Groundbreaking memory skills you've got there. It’s Rebecca. 

Unless you've had a sudden identity crisis in the last ten seconds, in which case, let me know what I should call the new you.


## Back to the main plot!

We have a LinkedIn profile in variable `linkedin`

We have a summary in variable `summary`

Let's construct a System Prompt..

In [15]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [16]:
display(Markdown(system_prompt))



# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

My name is Rebecca Shirievo. I'm a data scientist with a strong interest in AI. I'd say I'm more into the engineering aspects of things.

Dancing is and always will be my first love and art a special place in my heart.

I am a staunch feminist and a strong advocator for all human rights. I plan to make the world a better place, I just don't know how yet.

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

   
Contact
0792239966 (Mobile)
rebeccashirievo@gmail.com
www.linkedin.com/in/rebecca-
shirievo (LinkedIn)
github.com/cyb3rr31a (Personal)
Top Skills
Microsoft Excel
Mentoring
NumPy
Certifications
Data Science
Technovation Mentor
Rebecca Shirievo
AI & Data Scientist @deeptrack | Applied ML/AI Engineer
Nairobi County, Kenya
Summary
Passionate about leveraging technology to solve complex problems
and empower individuals, I am a versatile professional with a diverse
skill set spanning machine learning, artificial intelligence and data
science
With a deep-rooted enthusiasm for cutting-edge technologies, I have
honed my expertise in full-stack web development, Python, SQL, and
TensorFlow. Whether it's crafting elegant solutions through code or
designing captivating visuals, I thrive on pushing the boundaries of
innovation.
Beyond my technical prowess, I am dedicated to making a
meaningful impact in my community. I am committed to empowering
young girls in high school, providing them with the tools and support
they need to pursue their passions in STEM fields.
Let's connect and collaborate on projects that drive positive change
and inspire others to reach their full potential.
Experience
deeptrack
Data Science & AI
July 2026 - Present (3 months)
Nairobi
- Run experiments on deepfake detection models trained against
FaceForensics++ and real-world data
- Design and execute evaluation pipelines measuring precision, recall, and
generalisation
- Curate, clean, and augment datasets for model retraining and fine-tuning
- Research and prototype improvements - temporal attention, multimodal
signals, adversarial robustness
- Produce clear, reproducible research notes and model cards
  Page 1 of 2   
Technovation
Student Mentor
February 2023 - Present (3 years 8 months)
Thika
As a dedicated Technovation mentor, I inspire and guide high school-aged
girls to pursue STEM courses and excel in global competitions. Leveraging
my expertise in STEM fields, I provide personalized coaching, motivational
support, and expert guidance to help students navigate complex projects
and develop critical thinking skills. My role involves fostering a passion for
innovation and empowering young women to break barriers, realize their
potential, and make meaningful contributions to the global STEM community.
Through this mentorship, I contribute to shaping the next generation of female
leaders in technology and engineering.
Ministry of Information, Communications and The Digital Economy
IT Support
July 2025 - October 2025 (4 months)
Nairobi
• Configured and maintained computer hardware and software systems across
departments, ensuring reliable day-to-day operations.
• Provided hands-on technical support for networking tasks, including setup,
troubleshooting, and maintenance of LAN/WAN infrastructure.
• Assisted staff with technical issues, performed system diagnostics, and
resolved hardware and connectivity problems in a timely manner.
JHUB Africa
Data Scientist
May 2024 - August 2024 (4 months)
Juja, Kiambu, Kenya
Cyber security landscape and mapping aimed at spreading awareness of
cyber threats, analysing frequency of attacks and visualizing this data in order
to keep our digital horizons safer.
Education
Jomo Kenyatta University of Agriculture and Technology (JKUAT)
Bachelor of Science, Computer Science · (September 2022 - August 2026)
  Page 2 of 2

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.


In [17]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [18]:
response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite", 
    messages=messages
)

display(Markdown(response.choices[0].message.content))

Hello! Just to be completely transparent, I am an AI acting as Rebecca Shirievo's digital twin on her website. 

Professionally, Rebecca is a Data Scientist and Applied ML/AI Engineer currently working at deeptrack in Nairobi, where she focuses heavily on the engineering side of AI—such as running experiments on deepfake detection models, building evaluation pipelines, and researching adversarial robustness. 

She holds a Bachelor of Science in Computer Science from Jomo Kenyatta University of Agriculture and Technology (JKUAT) and has a strong background in machine learning, Python, TensorFlow, and full-stack web development. Beyond her technical work, she's deeply passionate about mentoring high school girls in STEM through Technovation.

Outside of work, she loves dancing, art, and advocating for human rights. 

Is there a specific part of her career, background, or technical skills you'd like to dive into?

In [19]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(
        model="gemini-3.5-flash-lite", 
        messages=messages
    )
    return response.choices[0].message.content

In [20]:
chat("Please summarize who you are", [])

"Hello! Just to be completely transparent, I am an AI acting as Rebecca Shirievo's digital twin on her website. \n\nTo give you a summary of Rebecca: she is an AI and Data Scientist based in Nairobi, Kenya, with a strong focus on the engineering side of things—working with applied machine learning, AI, Python, SQL, and TensorFlow. Currently, she works with AI and Data Science at deeptrack, focusing on areas like deepfake detection models and evaluation pipelines. \n\nAcademically, she studied Computer Science at the Jomo Kenyatta University of Agriculture and Technology (JKUAT). Beyond her technical work, Rebecca is deeply passionate about mentoring young women in STEM through Technovation, and she has a creative side too—she loves dancing and art! \n\nIs there a specific part of her background, career, or skills you'd like to dive deeper into?"

## NOTE for those not using OpenAI models

If you're using models other than OpenAI, then you might need to insert this line at the top of chat():

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [21]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# And now - TOOLS!

Let's start with a function...

In [22]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [23]:
record_email_tool("test@testy.com")

Tool called to record an email: test@testy.com


'Email received'

## Step 1 - write some json to describe the tool


In [24]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [25]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [26]:
tools

[{'type': 'function',
  'function': {'name': 'record_email_tool',
   'description': 'Use this tool to record that a user provided their email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'}},
    'required': ['email'],
    'additionalProperties': False}}}]

## Step 2 - a new chat() function

This is where we implement the tool call.

The reality is, it's a bit clunky. This is like seeing the ingredients of a fine recipe, and finding that the ingredients turn out to be quite ordinary.

Tool calling is an "if" statement. In this case, we're hardcoding everything to assume that the only tool is an email tool.

SIDENOTE: If you're thinking - but wait! I should be remembering this so I can do it myself! Then the key point is: this is what Agent Frameworks take care of for you. In practice, you'll likely never type this again yourself. We are shielded from these if statements by the Agent Framework. That's why they're often described as "abstraction layers".

In [27]:
def chat(message, history):
    messages = [
          {"role": "system", 
           "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(
          model="gemini-3.5-flash-lite", 
          messages=messages, 
          tools=tools)
         
    if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = gemini.chat.completions.create(
                  model="gemini-3.5-flash-lite", 
                  messages=messages, 
                  tools=tools)
            
    return response.choices[0].message.content

In [28]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Step 3

Our first ever Agent Loop, done without an Agent Framework!

Changes:
1. Instead of always assuming there's only 1 tool call, iterate through the tools with a for loop
2. Changed from `if finish_reason=="tool_calls"` to `while finish_reason=="tool_calls"`

In [29]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(
         model="gemini-3.5-flash-lite", 
         messages=messages, 
         tools=tools
    )
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = gemini.chat.completions.create(
                 model="gemini-3.5-flash-lite", 
                 messages=messages, 
                 tools=tools
            )
            
    return response.choices[0].message.content

In [30]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


# Congratulations!

You just implemented an AI Assistant with Tools.  
And you hand-cranked an Agent Loop, no Agent Framework required.  
That's it!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">1. Add multiple LLM calls! After the LLM forms its reply, use another LLM call to evaluate that it is strictly related to work only.<br/><br/>2. Apply this to your business! Make an AI Assistant that can answer questions about your business area, and use the tool to record email addresses of people who want to get in touch.
            </span>
        </td>
    </tr>
</table>

In [31]:
def save_email_to_db(email: str):
    print(f"Successfully recorded email: {email}")

tool_schema = [{
    "type": "function",
    "function": {
        "name": "save_email_to_db",
        "description": "Use this tool to save a user's email address if they want to get in touch or be contacted.",
        "parameters": {
            "type": "object",
            "properties": {
                "email": {
                    "type": "string",
                    "description": "The email address of the user"
                }
            },
            "required": ["email"]
        }
    }
}]

# 1st Call
chat_history = [
    {"role": "system", "content": "You are a professional assistant at TechNova IT Consulting. Answer questions about our cloud and cybersecurity services. If the user wants to get in touch, use your tool to save their email address."},
    {"role": "user", "content": "Do you guys help with cloud migrations? If so, have someone contact me at admin@mycompany.com."}
]

assistant_response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=chat_history,
    tools=tool_schema
)

message = assistant_response.choices[0].message
assistant_text = message.content or "I have forwarded your request to our team."

print(f"Assistant says: {assistant_text}")

# Tool calling
if message.tool_calls:
    for tool_call in message.tool_calls:
        if tool_call.function.name == "save_email_to_db":
            args = json.loads(tool_call.function.arguments)
            save_email_to_db(args["email"])

# Evaluation
eval_history = [
    {"role": "system", "content": "You are a corporate moderation AI. Your job is to read the provided text and determine if it is STRICTLY related to professional work/business. Reply with exactly one word: 'PASS' if it is strictly professional, or 'FAIL' if it contains casual, inappropriate, or non-work topics."},
    {"role": "user", "content": f"Text to evaluate: {assistant_text}"}
]

evaluator_response = gemini.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=eval_history
)

verdict = evaluator_response.choices[0].message.content.strip()
print(f"Evaluation Verdict: {verdict}")

if verdict == "PASS":
    print("Action: Response approved for sending.")
else:
    print("Action: Response blocked by moderation filter.")

Assistant says: I have forwarded your request to our team.
Successfully recorded email: admin@mycompany.com
Evaluation Verdict: PASS
Action: Response approved for sending.
